# 132 — MCP: tools, resources y prompts

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

**MCP**: protocolo abierto (Anthropic, 2024) que estandariza la frontera aplicación
LLM ↔ contexto: de m×n integraciones a m+n. Arquitectura **host → cliente (1:1) →
servidor**, mensajes **JSON-RPC 2.0**, ciclo `initialize` (negociación de
capacidades) → operación → cierre; transportes stdio y HTTP streaming.

**Tres primitivas del servidor, tres controladores**: **tools** (decide el *modelo*;
efectos; `tools/list` + `tools/call` con inputSchema), **resources** (decide la
*aplicación*; lectura por URI), **prompts** (decide el *usuario*; plantillas
parametrizadas).

**Errores en dos capas**: JSON-RPC (protocolo: tool inexistente) vs `isError: true`
(ejecución: el modelo lo lee y reacciona). Seguridad: consentimiento en el host,
servidores de terceros = código no confiable (tool poisoning).


## 🧮 Ejemplo de referencia

```text
→ initialize / ← capacidades      → tools/list
← [{name: "read_file", inputSchema: {required: ["path"]}}]
→ tools/call {name: "read_file", arguments: {path: "CHANGELOG.md"}}
← {content: [{type: "text", text: "## v2.41 ..."}], isError: false}
```

El laboratorio de esta clase (`run_lab("workflow")`) muestra la contraparte del host:
una máquina de estados que registra transiciones y aprobaciones — el andamiaje sobre
el que un host decide cuándo permitir un `tools/call` sensible.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("workflow", seed=132)
show(result)


## Reflexión

1. El laboratorio registra una aprobación antes de completar el workflow. ¿A cuál requisito de la spec de MCP (consentimiento en el host para tools sensibles) corresponde ese paso, y qué tool del ejemplo lo necesitaría?
2. ¿Por qué "archivo no encontrado" debe viajar como `isError: true` dentro de un result y no como error JSON-RPC? ¿Qué puede hacer el modelo en el primer caso que no puede en el segundo?
3. Si conviertes un resource (`file:///repo/README.md`) en una tool `read_readme()`, ¿qué decisión transferiste de la aplicación al modelo y qué riesgo nuevo aparece?
